In [216]:
# Train s_t on the unit sphere using the Monte-Carlo loss in your screenshot (Eq. (3))
# Specialized to:  M = S^{d-1} ⊂ R^d,  L(z;y) = -alpha * d_S(z,y)^2  (so exp(L)=exp(-alpha d^2))
#
# Requirements: pytorch (pip install torch)

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.func import vmap, jvp
import numpy as np
# -------------------------
# Sphere geometry (torch)
# -------------------------

def normalize_rows_torch(X: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    return X / (X.norm(dim=-1, keepdim=True).clamp_min(eps))

def tangent_project_torch(X: torch.Tensor, V: torch.Tensor) -> torch.Tensor:
    # P_X(V) = V - <X,V> X
    return V - (X * V).sum(dim=-1, keepdim=True) * X

def sphere_exp_map_torch(X: torch.Tensor, V: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    """
    X,V: (...,d) with V tangent at X
    """
    V = tangent_project_torch(X, V)
    r = V.norm(dim=-1, keepdim=True)
    sr_over_r = torch.where(r > eps, torch.sin(r) / r, 1.0 - (r*r)/6.0)
    Y = torch.cos(r) * X + sr_over_r * V
    return normalize_rows_torch(Y)

def sphere_distance(X: torch.Tensor, Y: torch.Tensor, eps: float = 1e-12, euclidean = True) -> torch.Tensor:
    """
    Geodesic distance on unit sphere using atan2(||proj||, dot).
    Supports broadcasting, e.g.
      X: (N,P,d), Y: (N,1,d)  -> (N,P)
      X: (d,),     Y: (N,d)   -> (N,)
    """
    # ensure tensors
    if not isinstance(X, torch.Tensor):
        X = torch.tensor(X)
    if not isinstance(Y, torch.Tensor):
        Y = torch.tensor(Y)

    # move Y to X's device/dtype
    Y = Y.to(device=X.device, dtype=X.dtype)

    # normalize
    Xn = X / X.norm(dim=-1, keepdim=True).clamp_min(eps)
    Yn = Y / Y.norm(dim=-1, keepdim=True).clamp_min(eps)

    # dot product
    dot = (Xn * Yn).sum(dim=-1).clamp(-1.0, 1.0)
    if euclidean:
        return torch.sqrt((2.0 * (1.0 - dot)).clamp_min(0.0))
    else:
        # sin(theta) = || X - (dot) Y ||  (works with broadcasting)
        proj = Xn - dot.unsqueeze(-1) * Yn
        sin_theta = proj.norm(dim=-1).clamp_min(0.0)

        return torch.atan2(sin_theta, dot)


def sphere_volume(dim: int) -> float:
    # vol(S^{dim-1}) = 2*pi^{dim/2} / Gamma(dim/2)
    return 2.0 * math.pi ** (dim / 2.0) / math.gamma(dim / 2.0)

# -------------------------------------------
# Geodesic random walk Brownian on the sphere
# -------------------------------------------

@torch.no_grad()
def brownian_endpoints_geodesic_rw(x: torch.Tensor, t: float, n_steps: int, n_paths: int, generator=None):
    """
    x: (d,) or (N,d)
    returns W: (N, n_paths, d)
    """
    x = normalize_rows_torch(x)

    if x.ndim == 1:
        x = x[None, :]          # (1,d)

    N, d = x.shape
    dt = t / n_steps

    X = x[:, None, :].expand(N, n_paths, d).contiguous()  # (N,P,d)

    for _ in range(n_steps):
        Z = torch.randn((N, n_paths, d), device=x.device, dtype=x.dtype, generator=generator)
        V = (dt ** 0.5) * Z
        V = tangent_project_torch(X, V)
        X = sphere_exp_map_torch(X, V)   # IMPORTANT: batch version
    return X


@torch.no_grad()
def qhat_t(
    x: torch.Tensor,
    y: torch.Tensor,
    t: float,
    n_steps: int,
    n_paths: int,
    alpha: float = 1.0,
    generator=None,
    euclidean = True
) -> torch.Tensor:
    """
    q_t(x) = E[ exp( -alpha * d(B_t, y)^2 ) | B_0=x ]
    Monte Carlo estimate at each x in batch.
    x: (N,d), y: (d,) or (N,d)
    returns: (N,)
    """
    x = normalize_rows_torch(x)
    if x.ndim == 1:
        x = x[None, :]

    y = y.to(device=x.device, dtype=x.dtype)
    y = normalize_rows_torch(y if y.ndim == 2 else y[None, :])
    if y.shape[0] == 1:
        yb = y.expand(x.shape[0], -1)  # (N,d)
    else:
        yb = y
    W = brownian_endpoints_geodesic_rw(x, t, n_steps, n_paths, generator=generator)  # (N,P,d)
    dist = sphere_distance(W, yb[:, None, :],euclidean=euclidean)  # (N,P)
    return torch.exp(-alpha * dist * dist).mean(dim=1)   # (N,)

# -------------------------------------------
# Neural net for s_t (ambient -> tangent)
# -------------------------------------------

class SphereScoreNet(nn.Module):
    """
    Residual MLP for a tangent vector field on S^2.

    - Input:  x ∈ R^3 (will be normalized to S^2)
    - Output: v(x) ∈ T_x S^2 (projected)
    """
    
    def __init__(self, d=3, hidden=256, depth=5, activation="silu", eps_norm=1e-18):
        super().__init__()
        assert depth >= 1
        self.eps_norm = float(eps_norm)

        act = {
            "silu": nn.SiLU(),
            "tanh": nn.Tanh(),
            "softplus": nn.Softplus(beta=1.0),
            "gelu": nn.GELU(),
        }[activation]

        layers = []
        # input -> hidden
        layers += [nn.Linear(d, hidden), act]

        # hidden -> hidden blocks
        for _ in range(depth - 1):
            layers += [nn.Linear(hidden, hidden), act]

        # hidden -> out
        layers += [nn.Linear(hidden, d)]

        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = normalize_rows_torch(x, eps=self.eps_norm)
        v = self.net(x)                 # ambient
        v = tangent_project_torch(x, v) # tangent
        return v
    
def div_hutchinson_jvp(
    model,
    X: torch.Tensor,          # (N,3)
    n_probes: int = 16,       # number of *pairs* if antithetic=True below
    eps_norm: float = 1e-18,
) -> torch.Tensor:
    """
    Estimate div_{S^2} s(X) where s is the tangent field returned by model.forward.

    Hutchinson with antithetic Rademacher probes:
        z entries iid in {+1,-1}, and we also use -z.
        tr(P J_s P) ≈ E_z [ z^T P J_s P z ], with z projected to T_X S^2.

    Uses torch.func.jvp + vmap, differentiable w.r.t model params.

    Args:
      n_probes: number of *base* probes; total probes used = 2*n_probes (z and -z).

    Returns:
      div: (N,)
    """
    X = normalize_rows_torch(X, eps=eps_norm)
    K = int(n_probes)

    # Base Rademacher probes (K,N,3)
    z = (2 * torch.randint(0, 2, (K,) + X.shape, device=X.device) - 1).to(dtype=X.dtype)

    # Antithetic pairing -> (2K,N,3)
    z = torch.cat([z, -z], dim=0)

    # Project probes to tangent at X
    z = tangent_project_torch(X.unsqueeze(0).expand_as(z), z)

    def one_probe(z_k: torch.Tensor) -> torch.Tensor:
        _, Js_z = jvp(model.forward, (X,), (z_k,))
        Js_z = tangent_project_torch(X, Js_z)
        return (z_k * Js_z).sum(dim=-1)  # (N,)

    div_est = vmap(one_probe, in_dims=0)(z)   # (2K, N)
    return div_est.mean(dim=0)                # (N,)

def loss_Lhat(
    model: nn.Module,
    X: torch.Tensor,
    y: torch.Tensor,
    t: float,
    n_steps_rw: int,
    n_paths_q: int,
    alpha: float,
    n_div_probes: int,
    div_eps: float,
    generator=None,
    euclidean = True
) -> torch.Tensor:
    X = normalize_rows_torch(X)
    y = normalize_rows_torch(y)

    s = model(X)                       # (N,d) tangent
    s_norm2 = (s * s).sum(dim=1)       # (N,)

    div_s = div_hutchinson_jvp(model, X, n_probes=n_div_probes, eps_norm=div_eps)  # (N,)

    with torch.no_grad():
        q = qhat_t(X, y, t=t, n_steps=n_steps_rw, n_paths=n_paths_q, alpha=alpha, generator=generator, euclidean=euclidean)  # (N,)

    vol = sphere_volume(X.shape[1])
    integrand = (0.5 * s_norm2 + div_s) * q
    return integrand.mean() / vol

# -------------------------------------------
# Training loop
# -------------------------------------------

def sample_uniform_sphere(N: int, d: int, device="cpu", generator=None) -> torch.Tensor:
    X = torch.randn((N, d), device=device, generator=generator)
    return normalize_rows_torch(X)

def train_score_on_sphere(
    y,
    t: float = 1,
    d: int = 3,
    alpha: float = 1.0,
    steps: int = 1000,
    batch_size: int = 256,
    lr: float = 1e-4,
    n_steps_rw: int = 5,
    n_paths_q: int = 80,
    n_div_probes: int = 4,
    div_eps: float = 1e-3,
    hidden: int = 256,
    depth: int = 4,
    device: str = "cpu",
    seed: int = 0,
    euclidean = True
):
    gen = torch.Generator(device=device)
    gen.manual_seed(seed)

    model = SphereScoreNet(d=d,hidden=hidden, depth=depth).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    # fixed y (you can also train with many y’s)

    for it in range(1, steps + 1):
        X = sample_uniform_sphere(batch_size, d, device=device, generator=gen)

        opt.zero_grad(set_to_none=True)
        L = loss_Lhat(
            model=model,
            X=X,
            y=y,
            t=t,
            n_steps_rw=n_steps_rw,
            n_paths_q=n_paths_q,
            alpha=alpha,
            n_div_probes=n_div_probes,
            div_eps=div_eps,
            generator=gen,
            euclidean = euclidean
        )
        L.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        if it % 100 == 0:
            print(f"iter {it:5d} | loss {L.item(): .6f}")
    return model

if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(device)



cpu


In [ ]:
import torch

@torch.no_grad()
def make_tangent_frame(X: torch.Tensor) -> torch.Tensor:
    """
    X: (B,d) on the unit sphere
    Returns U: (B,d,d-1) orthonormal tangent frames at each X[b].
    Deterministic: remove the most aligned coordinate axis, then QR.
    """
    X = normalize_rows_torch(X)
    B, d = X.shape
    device, dtype = X.device, X.dtype

    # Choose a column to drop per batch element (the axis most aligned with x)
    drop = torch.argmax(X.abs(), dim=1)  # (B,)

    # Build V = (I - x x^T) I = I - x x^T, shape (B,d,d)
    I = torch.eye(d, device=device, dtype=dtype).expand(B, d, d)
    xxT = X[:, :, None] * X[:, None, :]          # (B,d,d)
    V = I - xxT                                  # columns are tangent-projected basis vectors

    # Remove the "drop" column => (B,d,d-1)
    mask = torch.ones((B, d), device=device, dtype=torch.bool)
    mask.scatter_(1, drop[:, None], False)
    V_red = V[:, :, mask[0]].reshape(B, d, d-1)   # works because same mask pattern size; we’ll do per-batch gather below

    # Proper per-batch gather (safe)
    idx = torch.arange(d, device=device)[None, :].expand(B, d)          # (B,d)
    idx_keep = idx[mask].reshape(B, d-1)                                 # (B,d-1)
    V_red = V.gather(2, idx_keep[:, None, :].expand(B, d, d-1))          # (B,d,d-1)

    # Orthonormalize columns via QR
    Q, _ = torch.linalg.qr(V_red)                                        # (B,d,d-1)
    return Q



@torch.no_grad()
def fd_mc_gradlog_u_sphere_torch(
    x0: torch.Tensor,     # (B,d)
    y: torch.Tensor,      # (d,) or (B,d)
    t: float,
    alpha0: float,
    eps: float = 5e-2,
    n_steps: int = 200,
    n_paths: int = 20000,
    seed: int = 0,
    euclidean = True,
    return_u = True
):
    """
    Batched FD-MC score estimator on S^{d-1}.
    Needs your qhat_t(x, y, t, n_steps, n_paths, alpha, generator) -> (B,)

    Returns:
      u_hat: (B,) float tensor
      gradlog_hat: (B,d) float tensor tangent at x0
    """
    # ensure torch + device consistency
    if not isinstance(x0, torch.Tensor):
        x0 = torch.tensor(x0, dtype=torch.float32)
    if not isinstance(y, torch.Tensor):
        y = torch.tensor(y, dtype=torch.float32)

    device = x0.device
    dtype = x0.dtype
    y = y.to(device=device, dtype=dtype)

    gen = torch.Generator(device=device)
    gen.manual_seed(seed)

    x0 = normalize_rows_torch(x0)
    B, d = x0.shape
    n = d - 1

    # frames U0[b] in T_{x0[b]}
    U0 = make_tangent_frame(x0)  # (B,d,n)

    # base u(t,x0)
    u0 = qhat_t(x0, y, t, n_steps, n_paths, alpha=alpha0, generator=gen, euclidean=euclidean)  # (B,)

    # directional derivatives in intrinsic coordinates
    deriv = torch.zeros((B, n), device=device, dtype=dtype)

    # loop over tangent basis directions (d-1)
    for i in range(n):
        vi = U0[:, :, i]                        # (B,d)

        x_plus  = sphere_exp_map_torch(x0,  eps * vi)   # (B,d)
        x_minus = sphere_exp_map_torch(x0, -eps * vi)   # (B,d)

        u_plus  = qhat_t(x_plus,  y, t, n_steps, n_paths, alpha=alpha0, generator=gen)  # (B,)
        u_minus = qhat_t(x_minus, y, t, n_steps, n_paths, alpha=alpha0, generator=gen)  # (B,)

        deriv[:, i] = (u_plus - u_minus) / (2.0 * eps)

    # grad_u[b] = U0[b] @ deriv[b]
    grad_u = torch.einsum("bdn,bn->bd", U0, deriv)          # (B,d)
    grad_u = tangent_project_torch(x0, grad_u)

    gradlog = grad_u / u0.clamp_min(1e-30).unsqueeze(1)
    return (u0, gradlog) if return_u else gradlog 


In [159]:

import numpy as np
# ============================================================
# 1) Precompute b_l for f(theta)=exp(-alpha theta^2) on S^2
# ============================================================

def precompute_b_legendre_S2(alpha: float, Lmax: int, quad_n: int = 2000, device="cpu", dtype=torch.float64, euclidean = True):
    """
    b_l = (2l+1)/2 ∫_{-1}^1 exp(-alpha arccos(mu)^2) P_l(mu) dmu
    computed with Gauss-Legendre quadrature.
    Returns torch tensor b of shape (Lmax+1,) on device/dtype.
    """
    mu_np, w_np = np.polynomial.legendre.leggauss(quad_n)  # nodes/weights on [-1,1]
    mu = torch.tensor(mu_np, device=device, dtype=dtype)
    w  = torch.tensor(w_np,  device=device, dtype=dtype)

    if euclidean:
        #Euclidean distance 
        f = torch.exp(-alpha * (2.0 - 2.0 * mu))   # exp(-alpha ||x-y||^2) on unit sphere 
    else:
        #geodesic distance
        theta = torch.arccos(mu.clamp(-1.0, 1.0))
        f = torch.exp(-alpha * theta * theta)  # f(mu) = exp(-alpha arccos(mu)^2)

    # Build P_l(mu) table via recurrence: P shape (Lmax+1, Q)
    Q = mu.numel()
    P = torch.zeros((Lmax + 1, Q), device=device, dtype=dtype)
    P[0] = 1.0
    if Lmax >= 1:
        P[1] = mu
    for l in range(2, Lmax + 1):
        P[l] = ((2*l - 1) * mu * P[l-1] - (l - 1) * P[l-2]) / l

    # Integrals: ∫ f(mu) P_l(mu) dmu ≈ Σ w_q f_q P_l(mu_q)
    integ = (P * (w * f)[None, :]).sum(dim=1)  # (Lmax+1,)
    ell = torch.arange(Lmax + 1, device=device, dtype=dtype)
    b = (2*ell + 1) * 0.5 * integ
    return b


# ============================================================
# 2) Evaluate u(t,x) and score ∇ log u(t,x) on S^2
# ============================================================

@torch.no_grad()
def true_u_and_score_S2(
    X: torch.Tensor,         # (N,3)
    y: torch.Tensor,         # (3,)
    t: float,
    alpha: float,
    b: torch.Tensor,         # (Lmax+1,) from precompute
):
    """
    u(t,x) = Σ_l b_l e^{-0.5 l(l+1) t} P_l(mu),  mu=x·y
    ∇_x log u(t,x) = (u_mu / u) * (y - mu x)
    where u_mu = d/dmu u = Σ_l b_l e^{-...} P'_l(mu).

    Returns:
      u: (N,)
      score: (N,3) tangent at X
    """
    X = normalize_rows_torch(X)
    y = normalize_rows_torch(y.view(1, -1))[0]

    device, dtype = X.device, X.dtype
    Lmax = b.numel() - 1

    mu = (X @ y).clamp(-1.0, 1.0)   # (N,)

    # Build P_l(mu) for all mu via recurrence; shape (Lmax+1,N)
    N = mu.numel()
    P = torch.zeros((Lmax + 1, N), device=device, dtype=dtype)
    P[0] = 1.0
    if Lmax >= 1:
        P[1] = mu
    for l in range(2, Lmax + 1):
        P[l] = ((2*l - 1) * mu * P[l-1] - (l - 1) * P[l-2]) / l

    # P'_l(mu) using stable identity:
    # (1-mu^2) P'_l(mu) = l (P_{l-1}(mu) - mu P_l(mu))
    dP = torch.zeros_like(P)
    denom = (1.0 - mu*mu).clamp_min(1e-14)
    for l in range(1, Lmax + 1):
        dP[l] = l * (P[l-1] - mu * P[l]) / denom

    # Time decay coefficients
    ell = torch.arange(Lmax + 1, device=device, dtype=dtype)
    decay = torch.exp(-0.5 * ell * (ell + 1) * t)  # (Lmax+1,)
    c = b * decay                                   # (Lmax+1,)

    u = (c[:, None] * P).sum(dim=0)                 # (N,)
    u_mu = (c[:, None] * dP).sum(dim=0)             # (N,)

    # Tangent direction
    V = y[None, :] - mu[:, None] * X                # (N,3), tangent at X

    score = (u_mu / u.clamp_min(1e-30))[:, None] * V
    score = tangent_project_torch(X, score)
    return u, score


# ============================================================
# 3) Compare your model’s s_t(x) vs true score
# ============================================================

@torch.no_grad()
def score_metrics(pred: torch.Tensor, true: torch.Tensor):
    """
    pred,true: (N,3)
    Returns dict with MSE, mean cosine, relative L2 error.
    """
    mse = torch.mean((pred - true)**2).item()

    pred_norm = pred.norm(dim=1)
    true_norm = true.norm(dim=1)
    cos = (pred * true).sum(dim=1) / (pred_norm * true_norm).clamp_min(1e-12)
    mean_cos = cos.mean().item()

    rel = (pred - true).norm(dim=1) / true_norm.clamp_min(1e-12)
    rel_l2 = rel.mean().item()
    return {"mse": mse, "mean_cos": mean_cos, "mean_rel_l2": rel_l2}


# ============================================================
# Example evaluation
# ============================================================

def evaluate_model_vs_truth_S2(
    model,                 # your SphereScoreNet
    alpha: float,
    t: float,
    y: torch.Tensor,       # (3,)
    n_eval: int = 2048,
    Lmax: int = 400,
    quad_n: int = 2000,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
    euclidean = True
):
    model.eval().to(device)

    # Use higher precision for the spectral truth (helps near antipodes)
    dtype_truth = torch.float64

    # sample X ~ uniform on S^2
    X = torch.randn((n_eval, 3), device=device, dtype=torch.float32)
    X = normalize_rows_torch(X)

    y = y.to(device=device, dtype=torch.float32)
    y_truth = y.to(dtype=dtype_truth)

    # precompute b_l once
    b = precompute_b_legendre_S2(alpha=alpha, Lmax=Lmax, quad_n=quad_n, device=device, dtype=dtype_truth,euclidean=euclidean)

    # true score
    _, score_true = true_u_and_score_S2(X.to(dtype_truth), y_truth, t=t, alpha=alpha, b=b)
    score_true = score_true.to(dtype=torch.float32)

    # model score
    score_pred = model(X)

    # metrics
    metrics = score_metrics(score_pred, score_true)
    return metrics, X, score_pred, score_true


In [189]:
alpha = 1.0
t = 0.1
y = torch.Tensor([1,0,0])
model= train_score_on_sphere(
        y,
        d=3,
        t=t,
        alpha=1.0,
        steps=2000,
        batch_size=256,
        lr=1e-4,
        n_steps_rw=5,      # Brownian RW discretization steps
        n_paths_q= 80,      # MC paths for qhat_t per x
        hidden=256,
        n_div_probes =4,
        depth=4,
        device=device,
        seed=0,
        euclidean=True
    )

metrics, X, s_pred, s_true = evaluate_model_vs_truth_S2(
    model=model,
    alpha=alpha,
    t=t,
    y=y,
    n_eval=4096,
    Lmax=500,
    quad_n=2500,
    euclidean= True
)

print(metrics)

iter   100 | loss -0.014470
iter   200 | loss -0.025454
iter   300 | loss -0.013025
iter   400 | loss -0.017664
iter   500 | loss -0.014764
iter   600 | loss -0.019245
iter   700 | loss -0.015012
iter   800 | loss -0.016291
iter   900 | loss -0.013130
iter  1000 | loss -0.022651
iter  1100 | loss -0.014167
iter  1200 | loss -0.008832
iter  1300 | loss -0.025794
iter  1400 | loss -0.013896
iter  1500 | loss -0.015024
iter  1600 | loss -0.011384
iter  1700 | loss -0.023409
iter  1800 | loss -0.021061
iter  1900 | loss -0.022829
iter  2000 | loss -0.019845
{'mse': 0.0020425638649612665, 'mean_cos': 0.9995387196540833, 'mean_rel_l2': 0.05146150290966034}


In [211]:
import numpy as np
# ----------------------------
# S^2 geometry utilities
# ----------------------------

def normalize(x, eps=1e-12):
    x = np.asarray(x, dtype=float)
    nrm = np.linalg.norm(x, axis=-1, keepdims=True)
    return x / np.maximum(nrm, eps)

def tangent_project_np(x, v):
    return v - np.sum(x * v, axis=-1, keepdims=True) * x

def sphere_exp_map_np(x, v, eps=1e-12):
    # exp_x(v) on unit sphere (vectorized)
    x = normalize(x)
    v = tangent_project_np(x, v)
    r = np.linalg.norm(v, axis=-1, keepdims=True)
    sr_over_r = np.where(r > eps, np.sin(r) / r, 1.0 - (r**2)/6.0)
    y = np.cos(r) * x + sr_over_r * v
    return normalize(y)

def sphere_distance_np(x, y, euclidean = True):
    x = normalize(x)
    y = normalize(y)
    dot = np.sum(x * y, axis=-1)
    dot = np.clip(dot, -1.0, 1.0)
    if euclidean:
        return np.sqrt(np.maximum(2.0*(1.0 - dot), 0.0))  # chordal distance
    else:
        sin_theta = np.linalg.norm(x - dot[..., None] * y, axis=-1)
        return np.arctan2(sin_theta, dot)

def parallel_transport_S2(x, y, V):
    """
    Exact PT on S^2 along minimizing geodesic x->y.
    x,y: (...,3)
    V: (...,3,k) tangent vectors at x
    """
    x = normalize(x); y = normalize(y)
    dot = np.sum(x * y, axis=-1, keepdims=True)  # (...,1)
    denom = np.maximum(1.0 + dot, 1e-12)         # avoid antipodal blow-up

    yTV = np.sum(y[..., None] * V, axis=-2, keepdims=True)  # (...,1,k)
    corr = (yTV / denom[..., None]) * (x + y)[..., None]    # (...,3,k)
    W = V - corr
    # enforce tangency at y
    W = W - np.sum(y[..., None] * W, axis=-2, keepdims=True) * y[..., None]
    return W

def orthonormalize_frames(U):
    """U: (P,3,2) -> (P,3,2)"""
    P = U.shape[0]
    Q = np.zeros_like(U)
    # first
    v0 = U[:, :, 0]
    n0 = np.linalg.norm(v0, axis=1, keepdims=True)
    Q[:, :, 0] = v0 / np.maximum(n0, 1e-12)
    # second
    v1 = U[:, :, 1]
    proj = np.sum(Q[:, :, 0] * v1, axis=1, keepdims=True)
    v1 = v1 - proj * Q[:, :, 0]
    n1 = np.linalg.norm(v1, axis=1, keepdims=True)
    Q[:, :, 1] = v1 / np.maximum(n1, 1e-12)
    return Q

def make_tangent_frame_S2(x):
    """Return U0 (3,2) orthonormal columns spanning T_x S^2."""
    x = normalize(x.reshape(1,3))[0]
    # pick a helper not parallel to x
    a = np.array([1.0, 0.0, 0.0]) if abs(x[0]) < 0.9 else np.array([0.0, 1.0, 0.0])
    e1 = tangent_project_np(x, a)
    e1 = e1 / np.linalg.norm(e1)
    e2 = np.cross(x, e1)  # already tangent and unit
    U0 = np.stack([e1, e2], axis=1)  # (3,2)
    return U0

def normalize_rows_np(X, eps=1e-12):
    X = np.asarray(X, float)
    return X / np.linalg.norm(X, axis=-1, keepdims=True).clip(min=eps)

def tangent_project_np_batch(x, v):
    """
    Project v onto T_x S^2, batched.
    x: (B,3)
    v: (B,3)
    """
    return v - np.sum(x * v, axis=-1, keepdims=True) * x

def make_tangent_frame_S2_batch(X):
    """
    Vectorized version of make_tangent_frame_S2.
    Input:
      X: (B,3) (not necessarily normalized)
    Output:
      U: (B,3,2) with orthonormal columns spanning T_{x_b} S^2.
    """
    X = normalize_rows_np(X)               # (B,3)
    B = X.shape[0]

    # choose helper a per row:
    # if |x0| < 0.9 use (1,0,0) else (0,1,0)
    use_x = (np.abs(X[:, 0]) < 0.9)        # (B,)
    A = np.zeros((B, 3), dtype=float)
    A[use_x, 0] = 1.0
    A[~use_x, 1] = 1.0

    # e1 = proj_Tx(a), then normalize
    e1 = tangent_project_np_batch(X, A)
    e1 = normalize_rows_np(e1)             # (B,3)

    # e2 = x × e1 (already unit if x,e1 unit and orthogonal)
    e2 = np.cross(X, e1)                   # (B,3)
    # optional safety normalize (helps if numerical issues)
    e2 = normalize_rows_np(e2)

    U = np.stack([e1, e2], axis=-1)        # (B,3,2)
    return U


# ----------------------------
# BEL estimator (S^2)
# ----------------------------

def bel_gradlog_u_S2(
    x0, y, t, alpha,
    n_paths=20000,
    n_steps=200,
    seed=0,
    return_u_hat=True,
    euclidean = True
):
    """
    Estimates:
      u(t,x0) = E[ exp(-alpha d(X_t,y)^2) ]
      ∇ log u(t,x0) via Hsu BEL on S^2 (Ric=1 so M_s = exp(-s/2) I_2).

    Returns (u_hat, gradlog_hat) if return_u_hat else gradlog_hat.
    """
    rng = np.random.default_rng(seed)
    x0 = normalize(np.asarray(x0, float))
    y  = normalize(np.asarray(y, float))

    dt = t / n_steps
    U0 = make_tangent_frame_S2(x0)            # (3,2)

    # batch paths
    X = np.repeat(x0[None, :], n_paths, axis=0)        # (P,3)
    U = np.repeat(U0[None, :, :], n_paths, axis=0)     # (P,3,2)

    I = np.zeros((n_paths, 2), dtype=float)            # BEL Ito integral in R^2

    for k in range(n_steps):
        tk = (k + 0.5)* dt 
        M_scalar = np.exp(-0.5 * tk)   # Ric=1 -> M_t = exp(-t/2) I_2

        dW = rng.normal(size=(n_paths, 2)) * np.sqrt(dt)    # (P,2)
        V  = np.einsum("pij,pj->pi", U, dW)                 # (P,3)

        X_new = sphere_exp_map_np(X, V)                        # (P,3)
        U_new = parallel_transport_S2(X, X_new, U)          # (P,3,2)
        U_new = orthonormalize_frames(U_new)

        #U_new = make_tangent_frame_S2_batch(X_new)
        I += M_scalar * dW

        X, U = X_new, U_new

    dist = sphere_distance_np(X, y[None, :],euclidean=euclidean)          # (P,)
    fvals = np.exp(-alpha * dist * dist)                    # (P,)

    u_hat = float(np.mean(fvals))

    numer = np.sum(fvals[:, None] * I, axis=0)              # (2,)
    denom = t * np.sum(fvals) + 1e-30
    gradlog_intr = numer / denom                             # (2,)
    gradlog = U0 @ gradlog_intr                               # (3,)
    gradlog = gradlog - (gradlog @ x0) * x0                   # tangent

    return (u_hat, gradlog) if return_u_hat else gradlog

# ----------------------------
# Batched: many x's
# ----------------------------

def bel_gradlog_u_S2_batch(
    X0, y, t, alpha,
    n_paths=8000,
    n_steps=200,
    seed=0,
    euclidean=True
):
    """
    X0: (B,3) points. Returns:
      u_hat: (B,)
      gradlog_hat: (B,3)
    Uses independent MC for each x (simple & reliable).
    """
    X0 = normalize(np.asarray(X0, float))
    B = X0.shape[0]
    u = np.zeros(B, float)
    g = np.zeros((B,3), float)
    for i in range(B):
        u[i], g[i] = bel_gradlog_u_S2(
            X0[i], y, t, alpha,
            n_paths=n_paths,
            n_steps=n_steps,
            seed=seed + i,
            return_u_hat=True,
            euclidean=euclidean
        )
    return u, g

if __name__ == "__main__":
    x0 = np.array([0., 0., 1.])
    y  = np.array([1., 0., 0.])
    t = 0.2
    alpha = 5.0

    u_hat, gradlog_hat = bel_gradlog_u_S2(x0, y, t, alpha, n_paths=20000, n_steps=300, seed=0,euclidean=True)
    print("u_hat:", u_hat)
    print("gradlog_hat:", gradlog_hat)
    print("tangent check (dot x0):", float(gradlog_hat @ normalize(x0)))


u_hat: 0.007278219857017744
gradlog_hat: [4.68822572 0.01349554 0.        ]
tangent check (dot x0): 0.0


In [ ]:

# Make test points
rng = np.random.default_rng(0)
B = 250
X0 = rng.normal(size=(B,3))
X0 = X0 / np.linalg.norm(X0, axis=1, keepdims=True)
X_torch = torch.tensor(X0, dtype=torch.float32)
y = np.array([1., 0., 0.])
t = 1
alpha = 1
Lmax: int = 400
quad_n: int = 2000
dtype_truth = torch.float64
yt= torch.tensor(y)
yt = yt.to(device=device, dtype=torch.float32)
y_truth = yt.to(dtype=dtype_truth)

#Finite difference 
fd_grad_log = fd_mc_gradlog_u_sphere_torch(X_torch,y_truth,t, n_paths = 4000, n_steps=250,alpha0 = alpha,return_u=False,euclidean=True)

# BEL ground truth
_, BEL_g_true = bel_gradlog_u_S2_batch(X0, y, t, alpha, n_paths=4000, n_steps=250, seed=123,euclidean=True)

#sphere specific 
b = precompute_b_legendre_S2(alpha=alpha, Lmax=Lmax, quad_n=quad_n, device=device, dtype=dtype_truth,euclidean=True)
_, score_true = true_u_and_score_S2(X_torch.to(dtype_truth), y_truth, t=t, alpha=alpha, b=b)
score_true_np = score_true.to(dtype=torch.float32).numpy()


KeyboardInterrupt: 

In [161]:
# Model predictions
y = torch.Tensor([1,0,0])
model= train_score_on_sphere(
        y,
        d=3,
        t=t,
        alpha=1.0,
        steps=1000,
        batch_size=256,
        lr=1e-3,
        n_steps_rw=10,      # Brownian RW discretization steps
        n_paths_q= 1000,      # MC paths for qhat_t per x
        n_div_probes=32,     # Hutchinson probes for divergence
        hidden=256,
        depth=3,
        device=device,
        seed=0,
        euclidean=True
    )
with torch.no_grad():
    model_pred = model(X_torch).cpu()

iter   100 | loss -0.001548


KeyboardInterrupt: 

In [126]:
# Metrics
print('t = ', t)
print("NN", score_metrics(model_pred,score_true))
print("BEL", score_metrics(torch.tensor(BEL_g_true),score_true))
print("FD", score_metrics(fd_grad_log,score_true))

t =  1
NN {'mse': 0.012193795005364884, 'mean_cos': 0.9897995458801515, 'mean_rel_l2': 0.3739990461644834}
BEL {'mse': 0.018241895756185133, 'mean_cos': 0.9993002117846606, 'mean_rel_l2': 0.4451843795256935}
FD {'mse': 0.03411816543937326, 'mean_cos': 0.9473851634715625, 'mean_rel_l2': 0.6400308766266459}


EU t = 1
NN {'mse': 0.010828197589316087, 'mean_cos': 0.9916646634612438, 'mean_rel_l2': 0.3388261601135645}
BEL {'mse': 0.018241895756185133, 'mean_cos': 0.9993002117846606, 'mean_rel_l2': 0.4451843795256935}
FD {'mse': 0.03411816543937326, 'mean_cos': 0.9473851634715625, 'mean_rel_l2': 0.6400308766266459}

In [116]:
# pred,true: (N,3) numpy
model_pred_np = model_pred.numpy()
score_true_np = score_true.numpy()
num = np.sum(model_pred_np*score_true_np, axis=1)
den = np.sum(score_true_np*score_true_np, axis=1) + 1e-12
c_hat = num/den                 # per-point scale if pred ≈ c true
print("mean c_hat:", c_hat.mean(), "median:", np.median(c_hat))
print("mean |c_hat-1|:", np.mean(np.abs(c_hat-1)))

print("mean ||pred|| / ||true||:", np.mean(np.linalg.norm(model_pred_np,axis=1)/ (np.linalg.norm(score_true_np,axis=1)+1e-12)))



mean c_hat: 2.3711232612247914 median: 2.4658429356592557
mean |c_hat-1|: 1.4218773626588517
mean ||pred|| / ||true||: 2.375298498948386


In [235]:
def compare_models(T = np.array([0.01])):
    '''MSE and Cosin for the three models for different T'''
    ' for each row, euclidean MSE, cos, Geodesic MSE, Cos'
    BEL,NN,FD = np.zeros((len(T), 6)),np.zeros((len(T), 6)),np.zeros((len(T), 6))
    rng = np.random.default_rng(0)
    B = 500
    X0 = rng.normal(size=(B,3))
    X0 = X0 / np.linalg.norm(X0, axis=1, keepdims=True)
    X_torch = torch.tensor(X0, dtype=torch.float32, device=device)
    y = np.array([1., 0., 0.])
    alpha = 1
    Lmax: int = 500
    quad_n: int = 2500
    dtype_truth = torch.float64
    yt= torch.tensor(y)
    yt = yt.to(device=device, dtype=torch.float32)
    y_truth = yt.to(dtype=dtype_truth)
    #sphere constants
    b_eu = precompute_b_legendre_S2(alpha=alpha, Lmax=Lmax, quad_n=quad_n, device=device, dtype=dtype_truth,euclidean= True)
    b_geo = precompute_b_legendre_S2(alpha=alpha, Lmax=Lmax, quad_n=quad_n, device=device, dtype=dtype_truth,euclidean= False)
    for i in range(len(T)):
        t = T[i]

        #Polynomial approx specific 
        _, score_true_eu = true_u_and_score_S2(X_torch.to(dtype_truth), y_truth, t=t, alpha=alpha, b=b_eu)
        _, score_true_geo= true_u_and_score_S2(X_torch.to(dtype_truth), y_truth, t=t, alpha=alpha, b=b_geo)

        # BEL
        _, bel_eu = bel_gradlog_u_S2_batch(X0, y, t, alpha, n_paths=20000, n_steps=5, seed=123,euclidean=True)
        _, bel_geo = bel_gradlog_u_S2_batch(X0, y, t, alpha, n_paths=20000, n_steps=5, seed=123,euclidean=False)

        bel_eu_met = score_metrics(torch.tensor(bel_eu),score_true_eu)
        print('t = ', t, "Euclidean: BEL ",bel_eu_met)
        bel_geo_met = score_metrics(torch.tensor(bel_geo),score_true_geo)
        print('t = ', t, "Geodesic: BEL",bel_geo_met)

        BEL[i]=[bel_eu_met['mse'], bel_eu_met['mean_cos'],bel_eu_met['mean_rel_l2'],bel_geo_met['mse'], bel_geo_met['mean_cos'],bel_geo_met['mean_rel_l2']]

        #FD
        fd_eu = fd_mc_gradlog_u_sphere_torch(X_torch,y_truth,t=t,n_paths=20000, n_steps= 5, alpha0 = alpha,return_u=False,euclidean=True)
        fd_geo = fd_mc_gradlog_u_sphere_torch(X_torch,y_truth,t=t, n_paths=20000, n_steps= 5, alpha0 = alpha,return_u=False,euclidean=False)

        fd_eu_met = score_metrics(fd_eu,score_true_eu)
        print('t = ', t, "Euclidean: FD ",fd_eu_met)
        fd_geo_met = score_metrics(fd_geo,score_true_geo)
        print('t = ', t, "Geodesic: FD",fd_geo_met)

        FD[i]=[fd_eu_met['mse'], fd_eu_met['mean_cos'],fd_eu_met['mean_rel_l2'], fd_geo_met['mse'], fd_geo_met['mean_cos'],fd_geo_met['mean_rel_l2']]


        # Model predictions
        model_eu= train_score_on_sphere(
        y_truth,
        d=3,
        t=t,
        steps=10000,
        batch_size=256,
        lr=2e-3,
        n_steps_rw=5,      # Brownian RW discretization steps
        n_paths_q= 2000,      # MC paths for qhat_t per x
        hidden=256,
        n_div_probes =4,
        depth=4,
        device=device,
        seed=0,
        euclidean=True)
        nn_eu = model_eu(X_torch)
        nn_eu_met = score_metrics(nn_eu,score_true_eu)
        print('t = ', t, "Euclidean: NN ",nn_eu_met)
        
        model_geo= train_score_on_sphere(
        y_truth,
        d=3,
        t=t,
        steps=10000,
        batch_size=256,
        lr=2e-3,
        n_steps_rw=5,      # Brownian RW discretization steps
        n_paths_q= 2000,      # MC paths for qhat_t per x
        hidden=256,
        n_div_probes =4,
        depth=4,
        device=device,
        seed=0,
        euclidean=False)
        nn_geo = model_geo(X_torch)
        nn_geo_met = score_metrics(nn_geo,score_true_geo)
        print('t = ', t, "Geodesic: NN",nn_geo_met)

        NN[i]=[nn_eu_met['mse'], nn_eu_met['mean_cos'], nn_eu_met['mean_rel_l2'], nn_geo_met['mse'], nn_geo_met['mean_cos'],nn_geo_met['mean_rel_l2']]

    return BEL, FD, NN
        

In [236]:
BEL, FD, NN = compare_models()

t =  0.01 Euclidean: BEL  {'mse': 0.0033628353261120716, 'mean_cos': 0.9981504225495462, 'mean_rel_l2': 0.06794715542845622}
t =  0.01 Geodesic: BEL {'mse': 0.0038347007001481343, 'mean_cos': 0.999400098750356, 'mean_rel_l2': 0.03857709623865226}
t =  0.01 Euclidean: FD  {'mse': 0.00019659671535573905, 'mean_cos': 0.999948310712308, 'mean_rel_l2': 0.013159521065089212}
t =  0.01 Geodesic: FD {'mse': 17.4117360384591, 'mean_cos': 0.999948312111693, 'mean_rel_l2': 0.7375119263743805}
iter   100 | loss -0.009003


KeyboardInterrupt: 

In [227]:
BEL2, FD2, NN2 = BEL, FD, NN 

In [215]:
npaths = np.array([10, 50, 100, 250, 500, 1000, 4000, 8000, 12000, 16000, 20000, 24000, 28000, 32000])
T = np.array([1,0.1,0.01])
BEL_npath =  np.zeros((len(T),len(npaths), 3))
rng = np.random.default_rng(0)
B = 500
X0 = rng.normal(size=(B,3))
X0 = X0 / np.linalg.norm(X0, axis=1, keepdims=True)
X_torch = torch.tensor(X0, dtype=torch.float32, device=device)
y = np.array([1., 0., 0.])
alpha = 1
Lmax: int = 500
quad_n: int = 2500
dtype_truth = torch.float64
yt= torch.tensor(y)
yt = yt.to(device=device, dtype=torch.float32)
y_truth = yt.to(dtype=dtype_truth)
#sphere constants
b_eu = precompute_b_legendre_S2(alpha=alpha, Lmax=Lmax, quad_n=quad_n, device=device, dtype=dtype_truth,euclidean= True)
#Polynomial approx specific 
for j in range(len(T)):
    t= T[j]
    print("------------- t = ", t, "-------------")
    _, score_true_eu = true_u_and_score_S2(X_torch.to(dtype_truth), y_truth, t=t, alpha=alpha, b=b_eu)
    for i in range(len(npaths)):
        n_paths = npaths[i]
        # BEL
        _, bel_eu = bel_gradlog_u_S2_batch(X0, y, t, alpha, n_paths=n_paths, n_steps=5, seed=123,euclidean=True)

        bel_eu_met = score_metrics(torch.tensor(bel_eu),score_true_eu)
        print('n paths = ', n_paths, "BEL ",bel_eu_met)
        BEL_npath[j,i]=[bel_eu_met['mse'], bel_eu_met['mean_cos'],bel_eu_met['mean_rel_l2']]
    

------------- t =  1.0 -------------
n paths =  10 BEL  {'mse': 0.09207758693526788, 'mean_cos': 0.6925934425116156, 'mean_rel_l2': 0.9443853396677436}
n paths =  50 BEL  {'mse': 0.019248340250133444, 'mean_cos': 0.9150287146075636, 'mean_rel_l2': 0.4313402563212005}
n paths =  100 BEL  {'mse': 0.008994454435981047, 'mean_cos': 0.950576217632629, 'mean_rel_l2': 0.3040458256656809}
n paths =  250 BEL  {'mse': 0.003989719179580495, 'mean_cos': 0.9759047907267566, 'mean_rel_l2': 0.19544215329305897}
n paths =  500 BEL  {'mse': 0.0020530456437868848, 'mean_cos': 0.9874597275979123, 'mean_rel_l2': 0.1398819812065654}
n paths =  1000 BEL  {'mse': 0.000979042547107654, 'mean_cos': 0.9898966301249923, 'mean_rel_l2': 0.10155737121819905}
n paths =  4000 BEL  {'mse': 0.00021531701199371854, 'mean_cos': 0.9981672012742756, 'mean_rel_l2': 0.04744470744611022}
n paths =  8000 BEL  {'mse': 0.00011192981298056007, 'mean_cos': 0.9993385097728273, 'mean_rel_l2': 0.034756060075778035}
n paths =  12000 B